# LightGBM hyper-parameter tuning — validation period only

Starting point: the reference model `lgbm_otherdir` with the current `PARAMS` of `src/models/lgbm.py`.

Each configuration is a full walk-forward.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

import numpy as np
import pandas as pd

from src.data.data import build_auctions
from src.evaluation import Evaluator
from src.features.build_features import (
    LAG_FEATURES, OTHER_DIRECTION_FEATURES,
    add_lag_features, add_model_features, add_other_direction_features, build_features,
)
from src.models import ridge
from src.models.baseline import PreviousKnownAuctionBaseline
from src.models.lgbm import PARAMS
from src.models.tuning import CONFIGS, bidding_summary, run_configs, summarise

BEST = "background-color: #c6efce; color: #006100; font-weight: bold"
print("current PARAMS:", PARAMS)

current PARAMS: {'learning_rate': 0.05, 'num_leaves': 15, 'min_child_samples': 50, 'subsample': 0.8, 'subsample_freq': 1, 'colsample_bytree': 0.8, 'verbose': -1}


In [2]:
auctions_all = ridge.prepare_auctions(build_auctions(product="mFRR"), ("daily", "weekly"))
feats_ref = add_other_direction_features(
    add_lag_features(add_model_features(build_features(auctions_all))), auctions_all
)
FEATURE_COLS_REF = ridge.model_feature_columns(feats_ref) + LAG_FEATURES + OTHER_DIRECTION_FEATURES

VAL_START, VAL_LAST_DAY = "2022-01-01", "2024-12-31"     # same as lgbm.ipynb
VAL_END = (pd.Timestamp(VAL_LAST_DAY) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
in_val = (auctions_all["gc"] >= pd.Timestamp(VAL_START, tz="UTC")) & (auctions_all["gc"] < pd.Timestamp(VAL_END, tz="UTC"))
evaluator = Evaluator(actuals=auctions_all, start_test=VAL_START, end_test=VAL_END)

print("inputs:", FEATURE_COLS_REF)
print(VAL_START, "to", VAL_LAST_DAY, ":", in_val.sum(), "auctions")

inputs: ['lag1', 'lag2', 'lag3', 'block_00_04', 'block_04_08', 'block_08_12', 'block_12_16', 'block_16_20', 'block_20_24', 'daytype_saturday', 'daytype_sunday_holiday', 'daytype_workday', 'mom1', 'mom2', 'lag_mean', 'lag_std', 'other_lag1', 'other_gap']
2022-01-01 to 2024-12-31 : 13064 auctions


## 1. Round 1: one parameter at a time

| parameter | now | controls |
|---|---|---|
| `num_leaves` | 15 | complexity of each tree |
| `min_child_samples` | 50 | minimum auctions per leaf (higher = smoother, less confident quantiles) |
| `colsample_bytree` | 0.8 | share of the features each tree may use |
| `subsample` | 0.8 | share of the auctions each tree is built on |
| `reg_lambda` | 0 | penalty on large leaf values |

`learning_rate` stays at 0.05 (the number of trees is chosen on the grid 100 / 300 / 600 every month).

In [3]:
pd.DataFrame({"overrides": {name: str(o) for name, o in CONFIGS.items()}})

,overrides
default,{}
leaves_7,{'num_leaves': 7}
leaves_31,{'num_leaves': 31}
child_20,{'min_child_samples': 20}
child_100,{'min_child_samples': 100}
child_200,{'min_child_samples': 200}
colsample_05,{'colsample_bytree': 0.5}
colsample_10,{'colsample_bytree': 1.0}
subsample_06,{'subsample': 0.6}
lambda_1,{'reg_lambda': 1.0}


In [4]:
report, tuned = run_configs(CONFIGS, feats_ref, FEATURE_COLS_REF, evaluator, VAL_START, VAL_LAST_DAY)

default: running (current PARAMS)
leaves_7: running {'num_leaves': 7}
leaves_31: running {'num_leaves': 31}
child_20: running {'min_child_samples': 20}
child_100: running {'min_child_samples': 100}
child_200: running {'min_child_samples': 200}
colsample_05: running {'colsample_bytree': 0.5}
colsample_10: running {'colsample_bytree': 1.0}
subsample_06: running {'subsample': 0.6}
lambda_1: running {'reg_lambda': 1.0}
lambda_5: running {'reg_lambda': 5.0}


## 2. Compare

`mean`: mean pinball over the four groups (lower is better). `groups_better`: in how many groups the
configuration beats `default`. Green: best value of each column.

In [5]:
summary = summarise(report)
groups = [c for c in summary.columns if c not in ("mean", "groups_better")]
summary.style.highlight_min(axis=0, subset=groups + ["mean"], props=BEST).format(
    {**{c: "{:.4f}" for c in groups}, "mean": "{:.4f}", "groups_better": "{:.0f}"})

,daily downwards,daily upwards,weekly downwards,weekly upwards,mean,groups_better
config,,,,,,
default,0.0670,0.0757,0.0353,0.0136,0.0479,0
lambda_1,0.0671,0.0757,0.0353,0.0137,0.0480,0
colsample_05,0.0673,0.0758,0.0354,0.0136,0.0480,1
lambda_5,0.0673,0.0759,0.0353,0.0137,0.0480,0
subsample_06,0.0675,0.0757,0.0355,0.0135,0.0481,1
colsample_10,0.0673,0.0759,0.0353,0.0137,0.0481,0
leaves_31,0.0675,0.0758,0.0353,0.0136,0.0481,0
leaves_7,0.0675,0.0764,0.0353,0.0136,0.0482,0
child_100,0.0675,0.0759,0.0364,0.0139,0.0484,0


In [6]:
# coverage of the p10-p90 interval: target 0.80, green = closest to it in each column
coverage = report.pivot_table(index="config", columns=["freq", "direction"], values="coverage_80")

def closest_to_80(col):
    d = (col - 0.80).abs()
    return [BEST if v == d.min() else "" for v in d]

coverage.loc[summary.index].style.apply(closest_to_80, axis=0).format("{:.3f}")

In [7]:
# MAE of the prediction (p50), for reference
mae = report.pivot_table(index="config", columns=["freq", "direction"], values="mae")
mae.loc[summary.index].style.highlight_min(axis=0, props=BEST).format("{:.3f}")

## 3. Confirm with the bidding revenue

The 3 best configurations, based on the mean pinball loss, the default and the baseline. Mean revenue per year in CHF per MW
offered in every auction.

In [10]:
TOP = [c for c in summary.index if c != "default"][:3]
contracts = {"baseline": PreviousKnownAuctionBaseline().fit(auctions_all).predict(auctions_all[in_val])}
contracts.update({name: tuned[name] for name in ["default"] + TOP})

bids = bidding_summary(contracts, auctions_all)
revenue = bids.pivot_table(index=["freq", "direction"], columns=["model", "strategy"], values="revenue")
print("Mean revenue per year (CHF per MW offered)")
display(revenue.style.highlight_max(axis=1, props=BEST).format("{:,.0f}"))

share = bids.pivot_table(index=["freq", "direction"], columns=["model", "strategy"], values="share")
print("Share of the perfect revenue")
display(share.style.highlight_max(axis=1, props=BEST).format("{:.1%}"))

Mean revenue per year (CHF per MW offered)


Share of the perfect revenue


## Decision

- **Chosen configuration: `default`** 
- No configuration beats the default on the mean pinball loss (0.0479), and none is better in more
  than 1 of the 4 groups.
- The differences are very small so the model is not sensitive to these hyper-parameters..
- The gains of the project came from the features and, above all, from the bidding strategy, not from
  the hyper-parameters.